## *For up-to-date plotting script use plots/combineFiles.ipynb*

In [ ]:
import pandas as pd
import numpy as np
from coffea import util
import itertools
import os, sys
import matplotlib.pyplot as plt
import mplhep as hep
import uproot
import hist
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
hep.style.use("CMS")

sys.path.append('../python/')
from functions import loadCoffeaFile, getLabelMap, getCoffeaFilenames, plotBackgroundEstimate, getHist


## Scale factors and IOV

In [ ]:
# IOVs = ['2016APV', '2016', '2016all', '2017', '2018', 'Full']
IOVs = ['2016APV']

lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530./10.,
    "2018": 59800./10., #59740./10., #Blinding
    "Full": 46053. # 137190. Blinded
}

t_BR = 0.6741
ttbar_BR = 0.4544 #PDG 2019
ttbar_xs1 = 831.76 * (0.09210) #pb For ttbar mass from 700 to 1000
ttbar_xs2 = 831.76 * (0.02474) #pb For ttbar mass from 1000 to Inf
toptag_sf = 0.9
toptag_kf = 1.0 #0.7
qcd_xs = 1370000000.0 #pb From https://cms-gen-dev.cern.ch/xsdb



## make plot image filenames

In [ ]:
directories = [
    'images/png/closureTest/2016all',
    'images/png/closureTest/2016APV',
    'images/png/closureTest/2016',
    'images/png/closureTest/2017',
    'images/png/closureTest/2018',
    'images/png/closureTest/Full',
    'images/pdf/closureTest/2016all',
    'images/pdf/closureTest/2016APV',
    'images/pdf/closureTest/2016',
    'images/pdf/closureTest/2017',
    'images/pdf/closureTest/2018',
    'images/pdf/closureTest/Full',
    'images/png/kinematics/2016all',
    'images/png/kinematics/2016APV',
    'images/png/kinematics/2016',
    'images/png/kinematics/2017',
    'images/png/kinematics/2018',
    'images/png/kinematics/Full',
    'images/pdf/kinematics/2016all',
    'images/pdf/kinematics/2016APV',
    'images/pdf/kinematics/2016',
    'images/pdf/kinematics/2017',
    'images/pdf/kinematics/2018',
    'images/pdf/kinematics/Full'
]


for path in directories:
    if not os.path.exists(path):
        os.makedirs(path)

## functions

In [ ]:
def make_error_boxes(ax, xdata, ydata, xerror, yerror, facecolor='none',
                     edgecolor='none', alpha=0.5):
    
    # Loop over data points; create box from errors at each point
    errorboxes = [Rectangle((x - xe, y - ye), xe.sum(), ye.sum()) for x, y, xe, ye in zip(xdata, ydata, xerror.T, yerror.T)]

    # Create patch collection with specified colour/alpha
    pc = PatchCollection(errorboxes, facecolor=facecolor, alpha=alpha,
                         edgecolor=edgecolor)

    # Add collection to axes
    ax.add_collection(pc)

    # Plot errorbars
    artists = ax.errorbar(xdata, ydata, xerr=xerror, yerr=yerror,
                          fmt='none', ecolor='k', barsabove=True)

    return artists

def getHist(hname, ds, bkgest, year, sum_axes=[], integrate_axes={}, masspoint=''):
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames(False)
    
    cfiles = []
    sf = []
    bkgest_str = np.where([bkgest], 'weighted', 'unweighted')[0]
    
    for key, file in coffeaFiles[ds][bkgest_str][year].items():
        if masspoint != '':
            if masspoint in key:
                loaded_file = util.load(file)
                sum_axes_dict = {ax:sum for ax in sum_axes}
                histo = loaded_file[hname][integrate_axes][sum_axes_dict]
                histo = histo * (lumi[IOV] * 1.0 / loaded_file['cutflow']['sumw'])
                return histo
            
        loaded_file = util.load(file)
        cfiles.append(loaded_file)
        
        
        if 'TTbar' in ds and '700to1000' in key:
            sf.append(lumi[year] * ttbar_xs1 * toptag_sf**2 * toptag_kf / (loaded_file['cutflow']['sumw']))
        elif 'TTbar' in ds and '1000toInf' in key:
            sf.append(lumi[year] * ttbar_xs2 * toptag_sf**2 * toptag_kf / (loaded_file['cutflow']['sumw']))     
        elif 'QCD' in ds:
            sf.append(lumi[year] * qcd_xs / loaded_file['cutflow']['sumw'])  
        else:
            sf.append(1.)
        
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
    
    hists = []
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])    
    
    # sum all hists from dataset eras or pt bins
    histo = hists[0]*sf[0]
    if len(hists) > 1:
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]*sf[i+1]
            
            
    return histo

def getHistNoMassMod(hname, ds, year, sum_axes=[], integrate_axes={}):
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames()
    
    cfiles = []
    sf = []
    noMassMod_str = 'noMassMod'
    
    for key, file in coffeaFiles[ds][noMassMod_str][year].items():
            
        loaded_file = util.load(file)
        cfiles.append(loaded_file)
        
        
        if 'TTbar' in ds and '700to1000' in key:
            sf.append(lumi[year] * ttbar_xs1 * toptag_sf**2 * toptag_kf / loaded_file['cutflow']['sumw'])
        elif 'TTbar' in ds and '1000toInf' in key:
            sf.append(lumi[year] * ttbar_xs2 * toptag_sf**2 * toptag_kf / loaded_file['cutflow']['sumw'])  
        else:
            sf.append(1.)
        
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
    
    hists = []
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])    
    
    # sum all hists from dataset eras or pt bins
    histo = hists[0]*sf[0]
    if len(hists) > 1:
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]*sf[i+1]
            
            
    return histo
    
def plotBackgroundEstimate(HistDict, Text='', SaveFileName='', isInclusive=True, category='', signame='', xaxis='', linear=False, Pull=False, SignalPlot=False, RemoveBkg=False):
    
    Hbkg = HistDict['ntmj'] + HistDict['ttbar']
    Hbkg_fixed = HistDict['ntmj_fixed'] + HistDict['ttbar']
    Ndenom = np.sum(HistDict['antitag_data'].values() - HistDict['antitag_ttbar'].values())
#     Ndenom = HistDict['antitag_data'].values() - HistDict['antitag_ttbar'].values()
    mistag = HistDict['ntmj'] / HistDict['pretag'].values()
    term1 = 1. / HistDict['pretag'].values()
    term2 = (np.ones(len(mistag.values()))-mistag.values())/(Ndenom*mistag.values())
    errsNTMJ = HistDict['ntmj'].values()*np.sqrt( term1 + term2 )
    errs = np.sqrt( Hbkg.values() + errsNTMJ**2 )
    errs_stat = 1./HistDict['data'].values()
    height = errs * 2 # Assume symmetric error
    bottom = Hbkg_fixed.values() - errs
        
    if isInclusive and SaveFileName != '':
        with open(SaveFileName, 'a') as f:
            print('Obs.  = ',  '%10i'% np.sum(HistDict['data'].values()), ' +- ', np.sqrt(np.sum(HistDict['data'].values())), file=f)
            print('NMTJ  = ',  '%10i'% np.sum(HistDict['ntmj_fixed'].values()), ' +- ', np.sum(np.where(errsNTMJ>0., errsNTMJ, 0.)), file=f)
            print('TTbar = ',  '%10i'% np.sum(HistDict['ttbar'].values()), ' +- ', np.sqrt(np.sum(np.where(HistDict['ttbar'].values()>0., HistDict['ttbar'].values(), 0.))), file=f)
    elif not isInclusive and SaveFileName != '':
        with open(SaveFileName, 'a') as f:
            print(f'\t\t{category}\n===================================================', file=f)
            print('Obs.  = ',  '%10i'% np.sum(HistDict['data'].values()), ' +- ', np.sqrt(np.sum(HistDict['data'].values())), file=f)
            print('NMTJ  = ',  '%10i'% np.sum(HistDict['ntmj_fixed'].values()), ' +- ', np.sum(np.where(errsNTMJ>0., errsNTMJ, 0.)), file=f)
            print('TTbar = ',  '%10i'% np.sum(HistDict['ttbar'].values()), ' +- ', np.sqrt(np.sum(np.where(HistDict['ttbar'].values()>0., HistDict['ttbar'].values(), 0.))), file=f)
            print('\n', file=f)
    
    
#     centers = (edges[:-1] + edges[1:]) / 2
    
    if 'ttbarmass' in xaxis:
        edges = HistDict['ntmj_fixed'].axes['ttbarmass'].edges
    else:
        edges = HistDict['ntmj_fixed'].axes[xaxis].edges
    
    fig, (ax1, ax2) = plt.subplots(nrows=2, height_ratios=[3, 1])
    
    if '2016' in HistDict['IOV']:
        Year = '2016'
    else:
        Year = HistDict['IOV']

    if HistDict['IOV'] == 'Full':
        if linear:
            hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), loc=0, fontsize=15, ax=ax1)
        else:
            hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), loc=2, fontsize=20, ax=ax1)
            hep.cms.text(Text, loc=2, fontsize=20, ax=ax1)
    else:
        if linear:
            hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), year=Year, loc=0, fontsize=15, ax=ax1)
        else:
            hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), year=Year, loc=2, fontsize=20, ax=ax1)
            hep.cms.text(Text, loc=2, fontsize=20, ax=ax1)
            
    if RemoveBkg:
        SignalEst = HistDict['data'] + -1*Hbkg_fixed
        errs = np.sqrt( Hbkg.values() + HistDict['data'].values() + errsNTMJ**2 )
        height = errs * 2 # Assume symmetric error
        bottom = HistDict['ttbar'].values() - errs
        hep.histplot(HistDict['ttbar'],  ax=ax1, histtype='fill', color='xkcd:deep red', label='TTbar')
        hep.histplot(SignalEst, ax=ax1, histtype='errorbar', color='black', label='Data')
        ax1.bar(x = edges[:-1],
               height=height,
               bottom=bottom,
               width = np.diff(edges), align='edge', hatch='///', edgecolor='gray',
               linewidth=0, facecolor='none', alpha=0.8,
               zorder=10, label='Unc.')
    else:
        S = hist.Stack(HistDict['ttbar'], HistDict['ntmj_fixed'])
        S.plot(ax=ax1, stack=True, histtype="fill", color=['xkcd:deep red', 'xkcd:pale gold'], label=['TTbar', 'NTMJ'])
        hep.histplot(HistDict['data'],  ax=ax1, histtype='errorbar', color='black', label='Data')    
        ax1.bar(x = edges[:-1],
               height=height,
               bottom=bottom,
               width = np.diff(edges), align='edge', hatch='///', edgecolor='gray',
               linewidth=0, facecolor='none', alpha=0.8,
               zorder=10, label='Unc.')
    
    if SignalPlot:
        Signal = {
            'RSGluon' : r'RS$_{KK}$ Gluon',
            'ZPrime1' : r'Z ` $1\%$',
            'ZPrime10': r'Z ` $10\%$',
            'ZPrime30': r'Z ` $30\%$',
            'ZPrimeDM': r'Z ` DM'
        }
        hep.histplot(HistDict['sig1'], ax=ax1, histtype='step', label=Signal[signame]+' 1 TeV')
        hep.histplot(HistDict['sig4'], ax=ax1, histtype='step', ls='--', lw=3, label=Signal[signame]+' 4 TeV')
        
            
    if Pull != True:
        ratio_plot =  HistDict['data'] / Hbkg_fixed.values()
        ratioUnc = ratio_plot*np.sqrt( 1./(Hbkg_fixed.values()*ratio_plot.values()) + errs**2/(Hbkg_fixed.values()**2) )   #1 / errs

        ax2.bar(x = edges[:-1],
               height=(2.*ratioUnc.values()),
               bottom=(np.ones_like(ratio_plot.values()) - ratioUnc.values()),
               width = np.diff(edges), align='edge', edgecolor='black',
               linewidth=0, facecolor='red', alpha=0.3,
               zorder=10, label='Unc.')
        
        ax2.bar(x = edges[:-1],
               height=(2.*errs_stat),
               bottom=(np.ones_like(ratio_plot.values()) - errs_stat),
               width = np.diff(edges), align='edge', hatch='\\\\', edgecolor='black',
               linewidth=0, facecolor='blue', alpha=0.3,
               zorder=10, label='Stat. Unc.')
        
        hep.histplot(ratio_plot, yerr=ratioUnc.values(), ax=ax2, histtype='errorbar', color='black')
        
        legend2 = plt.legend(bbox_to_anchor =(0.1,-0.85), loc='lower center')
        ax2.set_ylim(0,2)
        ax2.axhline(1, color='black', ls='--')
        ax2.set_ylabel('Data/Bkg')
        
    elif Pull == True:
        if RemoveBkg:
            pull_plot = (SignalEst + -1*HistDict['ttbar']) / np.sqrt( SignalEst.values() + errs**2 ) 
            pull_plot_arr = np.where(SignalEst.values()>0., pull_plot.values(), np.nan)
        else:
            pull_plot = (HistDict['data'] + -1*Hbkg_fixed) / np.sqrt( HistDict['data'].values() + errs**2 ) 
            pull_plot_arr = np.where(HistDict['data'].values()>0., pull_plot.values(), np.nan)

        hist.plot.plot_pull_array(pull_plot, pull_plot_arr, ax=ax2, bar_kwargs={'color':'steelblue'}, pp_kwargs={'alpha':0.5})
                     
        ax2.set_ylim(-5,5)
        ax2.set_yticks([-4,-2,0,2,4])
        ax2.axhline(0, color='black', ls='--')
        ax2.set_ylabel(r'(D-B)/$\sigma$')

    ax1.legend(fontsize='xx-small', loc=1)
    ax1.set_yscale('log')
    ax1.set_ylabel('Events')
    ax1.set_xlabel('')
    ax1.set_ylim(1e-2, 1e7)
    ax1.set_xlim(900, 8000)
    ax2.set_xlim(900, 8000)    
    if linear:
        ax1.set_yscale('linear')
        ax1.autoscale('y')
        ax1.set_xlim(900, 8000)
        
    histname = HistDict['varname']
    if histname == 'jetpt':
        ax1.set_xlim(400, 2000)
        ax2.set_xlim(400, 2000)
    elif histname == 'jeteta':
        ax1.set_xlim(-2.4, 2.4)
        ax2.set_xlim(-2.4, 2.4)
    elif histname == 'jetphi':
        ax1.set_xlim(-np.pi, np.pi)
        ax2.set_xlim(-np.pi, np.pi)
    elif histname == 'sdjetmass':
        ax1.set_xlim(0, 500)
        ax2.set_xlim(0, 500)
    elif histname == 'jetp':
        ax1.set_xlim(400, 3600)
        ax2.set_xlim(400, 3600)
        
    if isInclusive:
        leg2 = plt.text(0.60, 0.60, 'b-tag Inclusive\n$|\Delta y|$ Inclusive',
                    fontsize=16,
                    weight='bold',
                    transform=ax1.transAxes
                   )
#     else: # ---- DELETE WHEN NOT TESTING WITH DEEPAK8 TAGGER ---- #
#         leg2 = plt.text(0.60, 0.60, 'DeepAK8 Top Tagger',
#                     fontsize=16,
#                     weight='bold',
#                     transform=ax1.transAxes
#                    )
        
        
def plotBackgroundEstimateComparison(HistDict, Text='', isInclusive=True, signame='', xaxis='', linear=False, Pull=False, SignalPlot=False):
    
    Hbkg = HistDict['ntmj'] + HistDict['ttbar']
    Hbkg_fixed = HistDict['ntmj_fixed'] + HistDict['ttbar']
    Hbkg_noMM = HistDict['ntmj_noMM'] + HistDict['ttbar']
    Hbkg_fixed_noMM = HistDict['ntmj_fixed_noMM'] + HistDict['ttbar']
    Ndenom = np.sum(HistDict['antitag_data'].values() - HistDict['antitag_ttbar'].values())

    mistag = HistDict['ntmj'] / HistDict['pretag'].values()
    mistag_noMM = HistDict['ntmj_noMM'] / HistDict['pretag'].values()
    term1 = 1. / HistDict['pretag'].values()
    term2 = (np.ones(len(mistag.values()))-mistag.values())/(Ndenom*mistag.values())
    term2_noMM = (np.ones(len(mistag_noMM.values()))-mistag_noMM.values())/(Ndenom*mistag_noMM.values())
    errsNTMJ = HistDict['ntmj'].values()*np.sqrt( term1 + term2 )
    errs = np.sqrt( Hbkg.values() + errsNTMJ**2 )
    errs_stat = 1./HistDict['data'].values()
    height = errs * 2 # Assume symmetric error
    bottom = Hbkg_fixed.values() - errs
    errsNTMJ_noMM = HistDict['ntmj_noMM'].values()*np.sqrt( term1 + term2_noMM )
    errs_noMM = np.sqrt( Hbkg_noMM.values() + errsNTMJ_noMM**2 )
    errs_stat_noMM = 1./HistDict['data'].values()
    height_noMM = errs_noMM * 2 # Assume symmetric error
    bottom_noMM = Hbkg_fixed_noMM.values() - errs_noMM
    
    
    if 'ttbarmass' in xaxis:
        edges = HistDict['ntmj_fixed'].axes['ttbarmass'].edges
    else:
        edges = HistDict['ntmj_fixed'].axes[xaxis].edges
    
    fig, ([ax1, bx1], [ax2, bx2]) = plt.subplots(
        nrows=2, 
        ncols=2, 
        figsize=(20,10),
        height_ratios=[3, 1])
    
    if '2016' in HistDict['IOV']:
        Year = '2016'
    else:
        Year = HistDict['IOV']

    if HistDict['IOV'] == 'Full':
        if linear:
            hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), loc=0, fontsize=15, ax=ax1)
            hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), loc=0, fontsize=15, ax=bx1)
        else:
            hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), loc=2, fontsize=20, ax=ax1)
            hep.cms.text(Text, loc=2, fontsize=20, ax=ax1)
            hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), loc=2, fontsize=20, ax=bx1)
            hep.cms.text(Text, loc=2, fontsize=20, ax=bx1)
    else:
        if linear:
            hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), year=Year, loc=0, fontsize=15, ax=ax1)
            hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), year=Year, loc=0, fontsize=15, ax=bx1)
        else:
            hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), year=Year, loc=2, fontsize=20, ax=ax1)
            hep.cms.text(Text, loc=2, fontsize=20, ax=ax1)
            hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[HistDict['IOV']]/1000.), year=Year, loc=2, fontsize=20, ax=bx1)
            hep.cms.text(Text, loc=2, fontsize=20, ax=bx1)
            
    S = hist.Stack(HistDict['ttbar'], HistDict['ntmj_fixed'])
    S_noMM = hist.Stack(HistDict['ttbar'], HistDict['ntmj_fixed_noMM'])
    S.plot(ax=ax1, stack=True, histtype="fill", color=['xkcd:deep red', 'xkcd:pale gold'], label=['TTbar', 'NTMJ'])
    S_noMM.plot(ax=bx1, stack=True, histtype="fill", color=['xkcd:deep red', 'xkcd:pale gold'], label=['TTbar', 'NTMJ'])
    hep.histplot(HistDict['data'],  ax=ax1, histtype='errorbar', color='black', label='Data')   
    hep.histplot(HistDict['data'],  ax=bx1, histtype='errorbar', color='black', label='Data')   
    ax1.bar(x = edges[:-1],
           height=height,
           bottom=bottom,
           width = np.diff(edges), align='edge', hatch='///', edgecolor='gray',
           linewidth=0, facecolor='none', alpha=0.8,
           zorder=10, label='Unc.')
    bx1.bar(x = edges[:-1],
           height=height_noMM,
           bottom=bottom_noMM,
           width = np.diff(edges), align='edge', hatch='///', edgecolor='gray',
           linewidth=0, facecolor='none', alpha=0.8,
           zorder=10, label='Unc.')
    
    if SignalPlot:
        Signal = {
            'RSGluon' : r'RS$_{KK}$ Gluon',
            'ZPrime1' : r'Z ` $1\%$',
            'ZPrime10': r'Z ` $10\%$',
            'ZPrime30': r'Z ` $30\%$',
            'ZPrimeDM': r'Z ` DM'
        }
        hep.histplot(HistDict['sig1'], ax=ax1, histtype='step', label=Signal[signame]+' 1 TeV')
        hep.histplot(HistDict['sig4'], ax=ax1, histtype='step', ls='--', lw=3, label=Signal[signame]+' 4 TeV')
        hep.histplot(HistDict['sig1'], ax=bx1, histtype='step', label=Signal[signame]+' 1 TeV')
        hep.histplot(HistDict['sig4'], ax=bx1, histtype='step', ls='--', lw=3, label=Signal[signame]+' 4 TeV')
        
            
    if Pull != True:
        ratio_plot =  HistDict['data'] / Hbkg_fixed.values()
        ratioUnc = ratio_plot*np.sqrt( 1./(Hbkg_fixed.values()*ratio_plot.values()) + errs**2/(Hbkg_fixed.values()**2) )
        ratio_plot_noMM =  HistDict['data'] / Hbkg_fixed_noMM.values()
        ratioUnc_noMM = ratio_plot_noMM*np.sqrt( 1./(Hbkg_fixed_noMM.values()*ratio_plot_noMM.values()) + errs_noMM**2/(Hbkg_fixed_noMM.values()**2) )

        ax2.bar(x = edges[:-1],
               height=(2.*ratioUnc.values()),
               bottom=(np.ones_like(ratio_plot.values()) - ratioUnc.values()),
               width = np.diff(edges), align='edge', edgecolor='black',
               linewidth=0, facecolor='red', alpha=0.3,
               zorder=10, label='Unc.')
        
        ax2.bar(x = edges[:-1],
               height=(2.*errs_stat),
               bottom=(np.ones_like(ratio_plot.values()) - errs_stat),
               width = np.diff(edges), align='edge', hatch='\\\\', edgecolor='black',
               linewidth=0, facecolor='blue', alpha=0.3,
               zorder=10, label='Stat. Unc.')
        
        bx2.bar(x = edges[:-1],
               height=(2.*ratioUnc_noMM.values()),
               bottom=(np.ones_like(ratio_plot_noMM.values()) - ratioUnc_noMM.values()),
               width = np.diff(edges), align='edge', edgecolor='black',
               linewidth=0, facecolor='red', alpha=0.3,
               zorder=10, label='Unc.')
        
        bx2.bar(x = edges[:-1],
               height=(2.*errs_stat_noMM),
               bottom=(np.ones_like(ratio_plot_noMM.values()) - errs_stat_noMM),
               width = np.diff(edges), align='edge', hatch='\\\\', edgecolor='black',
               linewidth=0, facecolor='blue', alpha=0.3,
               zorder=10, label='Stat. Unc.')
        
        hep.histplot(ratio_plot, yerr=ratioUnc.values(), ax=ax2, histtype='errorbar', color='black')
        hep.histplot(ratio_plot, yerr=ratioUnc_noMM.values(), ax=bx2, histtype='errorbar', color='black')
        
        legend2 = plt.legend(bbox_to_anchor =(0.1,-0.85), loc='lower center')
        ax2.set_ylim(0,2)
        ax2.axhline(1, color='black', ls='--')
        ax2.set_ylabel('Data/Bkg')
        bx2.set_ylim(0,2)
        bx2.axhline(1, color='black', ls='--')
        bx2.set_ylabel('Data/Bkg')
        
    elif Pull == True:
        
        pull_plot = (HistDict['data'] + -1*Hbkg_fixed) / np.sqrt( HistDict['data'].values() + errs**2 ) 
        pull_plot_arr = np.where(HistDict['data'].values()>0., pull_plot.values(), np.nan)
        pull_plot_noMM = (HistDict['data'] + -1*Hbkg_fixed_noMM) / np.sqrt( HistDict['data'].values() + errs_noMM**2 ) 
        pull_plot_arr_noMM = np.where(HistDict['data'].values()>0., pull_plot_noMM.values(), np.nan)

        hist.plot.plot_pull_array(pull_plot, pull_plot_arr, ax=ax2, bar_kwargs={'color':'steelblue'}, pp_kwargs={'alpha':0.5})
        hist.plot.plot_pull_array(pull_plot_noMM, pull_plot_arr_noMM, ax=bx2, bar_kwargs={'color':'steelblue'}, pp_kwargs={'alpha':0.5})
                     
        ax2.set_ylim(-5,5)
        ax2.set_yticks([-4,-2,0,2,4])
        ax2.axhline(0, color='black', ls='--')
        ax2.set_ylabel(r'(D-B)/$\sigma$')
        bx2.set_ylim(-5,5)
        bx2.set_yticks([-4,-2,0,2,4])
        bx2.axhline(0, color='black', ls='--')
        bx2.set_ylabel(r'(D-B)/$\sigma$')

    ax1.legend(fontsize='xx-small', loc=1)
    ax1.set_yscale('log')
    ax1.set_ylabel('Events')
    ax1.set_xlabel('')
    ax1.set_ylim(1e-2, 1e7)
    ax1.set_xlim(900, 8000)
    ax2.set_xlim(900, 8000)   
    
    bx1.legend(fontsize='xx-small', loc=1)
    bx1.set_yscale('log')
    bx1.set_ylabel('Events')
    bx1.set_xlabel('')
    bx1.set_ylim(1e-2, 1e7)
    bx1.set_xlim(900, 8000)
    bx2.set_xlim(900, 8000)   
    if linear:
        ax1.set_yscale('linear')
        ax1.autoscale('y')
        ax1.set_xlim(900, 8000)
        bx1.set_yscale('linear')
        bx1.autoscale('y')
        bx1.set_xlim(900, 8000)
        
    histname = HistDict['varname']
    if histname == 'jetpt':
        ax1.set_xlim(400, 2000)
        ax2.set_xlim(400, 2000)
        bx1.set_xlim(400, 2000)
        bx2.set_xlim(400, 2000)
    elif histname == 'jeteta':
        ax1.set_xlim(-2.4, 2.4)
        ax2.set_xlim(-2.4, 2.4)
        bx1.set_xlim(-2.4, 2.4)
        bx2.set_xlim(-2.4, 2.4)
    elif histname == 'jetphi':
        ax1.set_xlim(-np.pi, np.pi)
        ax2.set_xlim(-np.pi, np.pi)
        bx1.set_xlim(-np.pi, np.pi)
        bx2.set_xlim(-np.pi, np.pi)
    elif histname == 'sdjetmass':
        ax1.set_xlim(0, 500)
        ax2.set_xlim(0, 500)
        bx1.set_xlim(0, 500)
        bx2.set_xlim(0, 500)
    elif histname == 'jetp':
        ax1.set_xlim(400, 3600)
        ax2.set_xlim(400, 3600)
        bx1.set_xlim(400, 3600)
        bx2.set_xlim(400, 3600)
        
    if isInclusive:
        leg2a = plt.text(0.60, 0.60, 'Mass Mod\nb-tag Inclusive\n$|\Delta y|$ Inclusive',
                    fontsize=16,
                    weight='bold',
                    transform=ax1.transAxes
                   )
        leg2b = plt.text(0.60, 0.60, 'Without Mass Mod\nb-tag Inclusive\n$|\Delta y|$ Inclusive',
                    fontsize=16,
                    weight='bold',
                    transform=bx1.transAxes
                   )
    else:
        leg2a = plt.text(0.60, 0.60, 'Mass Mod',
                    fontsize=16,
                    weight='bold',
                    transform=ax1.transAxes
                   )
        leg2b = plt.text(0.60, 0.60, 'Without Mass Mod',
                    fontsize=16,
                    weight='bold',
                    transform=bx1.transAxes
                   )
    

        

def plotBackgroundEstimateNoData(histname, Hntmj, Httbar, Year, Text='', signame='', hsig1=None, hsig2=None, hsig3=None, hsig4=None):
    
    Hbkg = Hntmj + Httbar
    
    hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[Year]/1000.), year=Year, loc=2, fontsize=20)
    hep.cms.text(Text, loc=2, fontsize=20)

    hep.histplot(Hbkg, histtype='fill', color='xkcd:pale gold', label='NTMJ')
    hep.histplot(Httbar, histtype='fill', color='xkcd:deep red', label='TTbar')
    
    Signal = {
        'RSGluon' : r'RS$_{KK}$ Gluon',
        'ZPrime1' : r'Z ` $1\%$',
        'ZPrime10': r'Z ` $10\%$',
        'ZPrime30': r'Z ` $30\%$',
        'ZPrimeDM': r'Z ` DM'
    }
        
    if hsig1 != None:
        hep.histplot(hsig1, histtype='step', label=Signal['ZPrime1']+' 3 TeV')
        hep.histplot(hsig2, histtype='step', label=Signal['ZPrime10']+' 3 TeV')
        hep.histplot(hsig3, histtype='step', label=Signal['ZPrime30']+' 3 TeV')
        hep.histplot(hsig4, histtype='step', label=Signal['ZPrimeDM']+' 3 TeV')

    plt.legend(fontsize='xx-small')
    plt.yscale('log')
    plt.ylabel('Events')
    plt.xlabel('')
    plt.ylim(1e-2, 1e6)
    plt.xlim(900, 6000)
    plt.xlim(900, 6000)    
    

In [ ]:
def NeededHists(Variable, iov, signal, category=''):
    
    if category == '':
        signal_cats = [ i for label, i in label_to_int_dict.items() if '2t' in label]
        pretag_cats = [ i for label, i in label_to_int_dict.items() if 'pre' in label]
        anti_cats   = [ i for label, i in label_to_int_dict.items() if 'at' in label]
#         systs =       [ s for String, String in  ]
        
        if 'ttbarmass' in Variable:
            httbar = getHist(Variable, 'TTbar', False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam = getHist(Variable, 'TTbar', True, iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hcontam_noMM = getHistNoMassMod(Variable, 'TTbar', iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj = getHist(Variable, 'JetHT', True, iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_noMM = getHistNoMassMod(Variable, 'JetHT', iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata = getHist(Variable, 'JetHT', False, iov,  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag = getHist(Variable, 'JetHT', False, iov,sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'}) 
            hantitag_data = getHist(Variable, 'JetHT', False, iov, sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar = getHist(Variable, 'TTbar', False, iov, sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'}) 
            hsignal1000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')

        else:
            httbar = getHist(Variable, 'TTbar', False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam = getHist(Variable, 'TTbar', True, iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hcontam_noMM = getHistNoMassMod(Variable, 'TTbar', iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj = getHist(Variable, 'JetHT', True, iov,   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_noMM = getHistNoMassMod(Variable, 'JetHT', iov, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata = getHist(Variable, 'JetHT', False, iov,  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag = getHist(Variable, 'JetHT', False, iov,sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data = getHist(Variable, 'JetHT', False, iov,  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar = getHist(Variable, 'TTbar', False, iov,  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats}) 
            hsignal1000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000 = getHist(Variable, signal, False, iov, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
    else:
        signal_cats = label_to_int_dict['2t'+category]
        pretag_cats = label_to_int_dict['pret'+category]
        anti_cats   = label_to_int_dict['at'+category]
        
        if 'ttbarmass' in Variable:
            httbar = getHist(Variable, 'TTbar', False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam = getHist(Variable, 'TTbar', True, iov, sum_axes=[], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hcontam_noMM = getHistNoMassMod(Variable, 'TTbar', iov, sum_axes=[], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj = getHist(Variable, 'JetHT', True, iov,   sum_axes=[], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_noMM = getHistNoMassMod(Variable, 'JetHT', iov, sum_axes=[], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata = getHist(Variable, 'JetHT', False, iov,  sum_axes=[], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag = getHist(Variable, 'JetHT', False, iov,sum_axes=[], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data = getHist(Variable, 'JetHT', False, iov,  sum_axes=[], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar = getHist(Variable, 'TTbar', False, iov,  sum_axes=[], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'}) 
            hsignal1000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')

        else:
            httbar = getHist(Variable, 'TTbar', False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats})
            hcontam = getHist(Variable, 'TTbar', True, iov, sum_axes=[], integrate_axes={'anacat':pretag_cats})
            hcontam_noMM = getHistNoMassMod(Variable, 'TTbar', iov, sum_axes=[], integrate_axes={'anacat':pretag_cats})
            hntmj = getHist(Variable, 'JetHT', True, iov,   sum_axes=[], integrate_axes={'anacat':pretag_cats})
            hntmj_noMM = getHistNoMassMod(Variable, 'JetHT', iov, sum_axes=[], integrate_axes={'anacat':pretag_cats})
            hdata = getHist(Variable, 'JetHT', False, iov,  sum_axes=[], integrate_axes={'anacat':signal_cats})
            hpretag = getHist(Variable, 'JetHT', False, iov,sum_axes=[], integrate_axes={'anacat':pretag_cats})
            hantitag_data = getHist(Variable, 'JetHT', False, iov,  sum_axes=[], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar = getHist(Variable, 'TTbar', False, iov,  sum_axes=[], integrate_axes={'anacat':anti_cats}) 
            hsignal1000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000 = getHist(Variable, signal, False, iov, sum_axes=[], integrate_axes={'anacat':signal_cats}, masspoint='4000')
        
    neededHists = {
        'httbar': httbar,
        'hcontam': hcontam,
        'hntmj': hntmj,
        'hcontam_noMM': hcontam_noMM,
        'hntmj_noMM': hntmj_noMM,
        'hdata': hdata,
        'hpretag': hpretag,
        'hantitag_data': hantitag_data,
        'hantitag_ttbar': hantitag_ttbar,
        'hsignal1000': hsignal1000,
        'hsignal2000': hsignal2000,
        'hsignal3000': hsignal3000,
        'hsignal4000': hsignal4000
    }
    
    return neededHists

In [ ]:
def UseIOV(Variable, signal, category=''):
    
    Hists = NeededHists(Variable, IOV, signal, category)
    
    hntmj_fixed = Hists['hntmj'] + -1*Hists['hcontam']
    hntmj_fixed_noMM = Hists['hntmj_noMM'] + -1*Hists['hcontam_noMM']
    
    HistDict = {
        'varname': Variable,
        'pretag': Hists['hpretag'],
        'antitag_data': Hists['hantitag_data'],
        'antitag_ttbar': Hists['hantitag_ttbar'],
        'data': Hists['hdata'],
        'ntmj': Hists['hntmj'],
        'ntmj_fixed': hntmj_fixed,
        'ntmj_noMM': Hists['hntmj_noMM'],
        'ntmj_fixed_noMM': hntmj_fixed_noMM,
        'ttbar': Hists['httbar'],
        'sig1': Hists['hsignal1000'],
        'sig2': Hists['hsignal2000'],
        'sig3': Hists['hsignal3000'],
        'sig4': Hists['hsignal4000'],
        'IOV': IOV
    }
    return HistDict

In [ ]:
def Use2016allIOV(Variable, signal, category=''):
    

    HistsAPV = NeededHists(Variable, '2016APV', signal, category)
    HistsnoAPV = NeededHists(Variable, '2016', signal, category)
    
    httbar  = HistsAPV['httbar'] + HistsnoAPV['httbar'] 
    hcontam = HistsAPV['hcontam'] + HistsnoAPV['hcontam'] 
    hntmj   = HistsAPV['hntmj'] + HistsnoAPV['hntmj'] 
    hcontam_noMM = HistsAPV['hcontam_noMM'] + HistsnoAPV['hcontam_noMM'] 
    hntmj_noMM   = HistsAPV['hntmj_noMM'] + HistsnoAPV['hntmj_noMM'] 
    hdata   = HistsAPV['hdata'] + HistsnoAPV['hdata'] 
    hpretag = HistsAPV['hpretag'] + HistsnoAPV['hpretag'] 
    hantitag_data = HistsAPV['hantitag_data'] + HistsnoAPV['hantitag_data'] 
    hantitag_ttbar = HistsAPV['hantitag_ttbar'] + HistsnoAPV['hantitag_ttbar'] 
    hsignal1000 = HistsAPV['hsignal1000'] + HistsnoAPV['hsignal1000']
    hsignal2000 = HistsAPV['hsignal2000'] + HistsnoAPV['hsignal2000']
    hsignal3000 = HistsAPV['hsignal3000'] + HistsnoAPV['hsignal3000']
    hsignal4000 = HistsAPV['hsignal4000'] + HistsnoAPV['hsignal4000']
    
    hntmj_fixed = hntmj + -1*hcontam
    hntmj_fixed_noMM = hntmj_noMM + -1*hcontam_noMM
    
    HistDict = {
        'varname': Variable,
        'pretag': hpretag,
        'antitag_data': hantitag_data,
        'antitag_ttbar': hantitag_ttbar,
        'data': hdata,
        'ntmj': hntmj,
        'ntmj_fixed': hntmj_fixed,
        'ntmj_noMM': hntmj_noMM,
        'ntmj_fixed_noMM': hntmj_fixed_noMM,
        'ttbar': httbar,
        'sig1': hsignal1000,
        'sig2': hsignal2000,
        'sig3': hsignal3000,
        'sig4': hsignal4000,
        'IOV': '2016all'
    }
    return HistDict

In [ ]:
def UseFullIOV(Variable, signal, category=''):
    
    HistsAPV = NeededHists(Variable, '2016APV', signal, category)
    HistsnoAPV = NeededHists(Variable, '2016', signal, category)
    Hists17 = NeededHists(Variable, '2017', signal, category)
    Hists18 = NeededHists(Variable, '2018', signal, category)
    
    httbar  = HistsAPV['httbar'] + HistsnoAPV['httbar'] + Hists17['httbar'] + Hists18['httbar']
    hcontam = HistsAPV['hcontam'] + HistsnoAPV['hcontam'] + Hists17['hcontam'] + Hists18['hcontam']
    hntmj   = HistsAPV['hntmj'] + HistsnoAPV['hntmj'] + Hists17['hntmj'] + Hists18['hntmj']
    hcontam_noMM = HistsAPV['hcontam_noMM'] + HistsnoAPV['hcontam_noMM'] + Hists17['hcontam_noMM'] + Hists18['hcontam_noMM']
    hntmj_noMM   = HistsAPV['hntmj_noMM'] + HistsnoAPV['hntmj_noMM'] + Hists17['hntmj_noMM'] + Hists18['hntmj_noMM']
    hdata   = HistsAPV['hdata'] + HistsnoAPV['hdata'] + Hists17['hdata'] + Hists18['hdata']
    hpretag = HistsAPV['hpretag'] + HistsnoAPV['hpretag'] + Hists17['hpretag'] + Hists18['hpretag']
    hantitag_data = HistsAPV['hantitag_data'] + HistsnoAPV['hantitag_data'] + Hists17['hantitag_data'] + Hists18['hantitag_data']
    hantitag_ttbar = HistsAPV['hantitag_ttbar'] + HistsnoAPV['hantitag_ttbar'] + Hists17['hantitag_ttbar'] + Hists18['hantitag_ttbar']
    hsignal1000 = HistsAPV['hsignal1000'] + HistsnoAPV['hsignal1000'] + Hists17['hsignal1000'] + Hists18['hsignal1000']
    hsignal2000 = HistsAPV['hsignal2000'] + HistsnoAPV['hsignal2000'] + Hists17['hsignal1000'] + Hists18['hsignal1000']
    hsignal3000 = HistsAPV['hsignal3000'] + HistsnoAPV['hsignal3000'] + Hists17['hsignal1000'] + Hists18['hsignal1000']
    hsignal4000 = HistsAPV['hsignal4000'] + HistsnoAPV['hsignal4000'] + Hists17['hsignal1000'] + Hists18['hsignal1000']
    
    hntmj_fixed = hntmj + -1*hcontam
    hntmj_fixed_noMM = hntmj_noMM + -1*hcontam_noMM
    
    HistDict = {
        'varname': Variable,
        'pretag': hpretag,
        'antitag_data': hantitag_data,
        'antitag_ttbar': hantitag_ttbar,
        'data': hdata,
        'ntmj': hntmj,
        'ntmj_fixed': hntmj_fixed,
        'ntmj_noMM': hntmj_noMM,
        'ntmj_fixed_noMM': hntmj_fixed_noMM,
        'ttbar': httbar,
        'sig1': hsignal1000,
        'sig2': hsignal2000,
        'sig3': hsignal3000,
        'sig4': hsignal4000,
        'IOV': 'Full'
    }
    return HistDict

In [ ]:
# analysis categories #
label_dict = util.load(f'../outputs/QCD_2016_noSyst.coffea')['analysisCategories']
label_to_int_dict = {label: i for i, label in label_dict.items()}
print(label_dict)

## plot background estimate (inclusive)

In [ ]:
variable = 'ttbarmass'  # Change this at will
signal = 'ZPrime30'
useSignal = False
usePull = True
Linear = True
# removeBkg = False
linearStr = ''
sigstr = ''
SaveFileName = '../data/BkgEstEventCounts_Inc.txt'

# -- prepare to overwrite event-count txt -- #
file_to_delete = open(SaveFileName,'w')
file_to_delete.close()

if Linear:
    linearStr = '_LINEAR'
histDict = {}

dirname = 'closureTest'
if variable != 'ttbarmass':
    dirname = 'kinematics'
if useSignal:
    sigstr = '_with_' + signal
    
    
for IOV in IOVs:

    if 'Full' in IOV:
        histDict = UseFullIOV(variable, signal)
    elif '2016all' in IOV:
        histDict = Use2016allIOV(variable, signal)
    else:
        histDict = UseIOV(variable, signal)

    if Linear:
        text = f'Preliminary'
    else:
        text = f'Preliminary\n'

    plotBackgroundEstimate(histDict, text, SaveFileName, True, '', signal, variable, Linear, usePull, useSignal)
        
    if usePull:
        savefilename = f'images/png/kinematics/{IOV}/{variable}{sigstr}_Inclusive{linearStr}.png'
    else:
        savefilename = f'images/png/{dirname}/{IOV}/{variable}{sigstr}_Inclusive{linearStr}.png'

    print(savefilename)
#     plt.savefig(savefilename)
#     plt.savefig(savefilename.replace('png', 'pdf'))

    plt.show()

## plot background estimate (by category)

In [ ]:
variable = 'ttbarmass'  # Change this at will
signal = 'ZPrime1'
useSignal = False
usePull = True
Linear = True
linearStr = ''
sigstr = ''

SaveFileName = '../data/BkgEstEventCounts_Categories.txt'

# -- prepare to overwrite event-count txt -- #
file_to_delete = open(SaveFileName,'w')
file_to_delete.close()

if Linear:
    linearStr = '_LINEAR'
histDict = {}

dirname = 'closureTest'
if variable != 'ttbarmass':
    dirname = 'kinematics'
if useSignal:
    sigstr = '_with_' + signal
    
for IOV in IOVs:
    
    for cat in ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']:

        if 'Full' in IOV:
            histDict = UseFullIOV(variable, signal, cat)
        elif '2016all' in IOV:
            histDict = Use2016allIOV(variable, signal, cat)
        else:
            histDict = UseIOV(variable, signal, cat)
            
        dytext = ''
        if 'cen' in cat:
            dytext = r'$\Delta y$ < 1.0'
        elif 'fwd' in cat:
            dytext = r'$\Delta y$ > 1.0'

        btext = ''
        if '0b' in cat:
            btext = '0 b-tags'
        elif '1b' in cat:
            btext = '1 b-tag'
        elif '2b' in cat:
            btext = '2 b-tags'

        if Linear:
            text = f'Preliminary:    {btext}, {dytext}'
        else:
            text = f'Preliminary\n{btext}, {dytext} \n'

        plotBackgroundEstimate(histDict, text, SaveFileName, False, cat, signal, variable, Linear, usePull, useSignal)

        if usePull:
            savefilename = f'images/png/kinematics/{IOV}/{variable}{sigstr}_{cat}{linearStr}.png'
        else:
            savefilename = f'images/png/{dirname}/{IOV}/{variable}{sigstr}_{cat}{linearStr}.png'

        print(savefilename)
#         plt.savefig(savefilename)
#         plt.savefig(savefilename.replace('png', 'pdf'))

        plt.show()

# Comparison without Mass Modification (Inclusive)

In [ ]:
variable = 'ttbarmass'  # Change this at will
signal = 'ZPrime30'
useSignal = False
usePull = True
Linear = True
# removeBkg = False
linearStr = ''
sigstr = ''
if Linear:
    linearStr = '_LINEAR'
histDict = {}

dirname = 'closureTest'
if variable != 'ttbarmass':
    dirname = 'kinematics'
if useSignal:
    sigstr = '_with_' + signal
    
    
for IOV in IOVs:

    if 'Full' in IOV:
        histDict = UseFullIOV(variable, signal)
    elif '2016all' in IOV:
        histDict = Use2016allIOV(variable, signal)
    else:
        histDict = UseIOV(variable, signal)

    if Linear:
        text = f'Preliminary'
    else:
        text = f'Preliminary\n'

    plotBackgroundEstimateComparison(histDict, text, True, signal, variable, Linear, usePull, useSignal)
        
    if usePull:
        savefilename = f'images/png/kinematics/{IOV}/{variable}{sigstr}_Inclusive{linearStr}.png'
    else:
        savefilename = f'images/png/{dirname}/{IOV}/{variable}{sigstr}_Inclusive{linearStr}.png'
        

    print(savefilename)
#     plt.savefig(savefilename)
#     plt.savefig(savefilename.replace('png', 'pdf'))

    plt.show()

# Comparison without Mass Modification (Categories)

In [ ]:
variable = 'ttbarmass'  # Change this at will
signal = 'ZPrime1'
useSignal = False
usePull = True
Linear = True
linearStr = ''
sigstr = ''
if Linear:
    linearStr = '_LINEAR'
histDict = {}

dirname = 'closureTest'
if variable != 'ttbarmass':
    dirname = 'kinematics'
if useSignal:
    sigstr = '_with_' + signal
    
for IOV in IOVs:
    
    for cat in ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']:

        if 'Full' in IOV:
            histDict = UseFullIOV(variable, signal, cat)
        elif '2016all' in IOV:
            histDict = Use2016allIOV(variable, signal, cat)
        else:
            histDict = UseIOV(variable, signal, cat)
            
        dytext = ''
        if 'cen' in cat:
            dytext = r'$\Delta y$ < 1.0'
        elif 'fwd' in cat:
            dytext = r'$\Delta y$ > 1.0'

        btext = ''
        if '0b' in cat:
            btext = '0 b-tags'
        elif '1b' in cat:
            btext = '1 b-tag'
        elif '2b' in cat:
            btext = '2 b-tags'

        if Linear:
            text = f'Preliminary:    {btext}, {dytext}'
        else:
            text = f'Preliminary\n{btext}, {dytext} \n'

        plotBackgroundEstimateComparison(histDict, text, False, signal, variable, Linear, usePull, useSignal)

        if usePull:
            savefilename = f'images/png/kinematics/{IOV}/{variable}{sigstr}_{cat}{linearStr}.png'
        else:
            savefilename = f'images/png/{dirname}/{IOV}/{variable}{sigstr}_{cat}{linearStr}.png'

        print(savefilename)
#         plt.savefig(savefilename)
#         plt.savefig(savefilename.replace('png', 'pdf'))

        plt.show()

In [ ]:
# Signals = {
#     'ZPrime1' : ['1000', '2000', '3000', '4000'],
#     'ZPrime10': ['1000', '2000', '3000', '4000'],
#     'ZPrime30': ['1000', '2000', '3000', '4000'],
#     'ZPrimeDM': ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000'],
#     'RSGluon':  ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000']
# }

# signal = 'ZPrime30'

# IOV = '2018'

# cats = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']
# cat_labels = ['cen0b', 'fwd0b', 'cen1b', 'fwd1b', 'cen2b', 'fwd2b']

# systematics = ['nominal', 'jes', 'jer', 'pileup', 'pdf', 'q2', 'btag']#, 'prefiring']
# syst_labels = ['nominal']
# for s in systematics:
#     if not 'nominal' in s:
#         syst_labels.append(s+'Down')
#         syst_labels.append(s+'Up')
        
# print(syst_labels)

# savefileheader = '../outputs/combine/categories/TTbarAllHad{}_'.format(IOV.replace('20', '').replace('all',''))
# print(savefileheader)

# froot = uproot.recreate(savefileheader+'CombineRoot_Cat.root')

# variable = 'ttbarmass'



# # for cat, catname in zip(cats, cat_labels):
# #     signal_cat = label_to_int_dict['2t'+cat]
# #     pretag_cat = label_to_int_dict['pret'+cat]
# #     httbar_trial = getHist(variable, 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat})
# #     print(httbar_trial)



# # for IOV in IOVs:

# for cat, catname in zip(cats, cat_labels):

#     signal_cat = label_to_int_dict['2t'+cat]
#     pretag_cat = label_to_int_dict['pret'+cat]

#     hsignal = {}
#     hsignal['ZPrime1'] = {}
#     hsignal['ZPrime10'] = {}
#     hsignal['ZPrime30'] = {}
#     hsignal['ZPrimeDM'] = {}
#     hsignal['RSGluon'] = {}

#     for syst in syst_labels:

#         if 'Full' in IOV:

#             catsystString = catname+'_'+syst

#             httbar_apv        = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_apv       = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_noapv      = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_noapv     = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_17      = getHist(variable, 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_17     = getHist(variable, 'TTbar', True, '2017', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_18      = getHist(variable, 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_18     = getHist(variable, 'TTbar', True, '2018', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

#             print('loading signals...')
#             for sig in Signals.keys():
#                 for mass in Signals[sig]:
#                     hsignal_apv   = getHist(variable, sig, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_noapv = getHist(variable, sig, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_17 = getHist(variable, sig, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_18 = getHist(variable, sig, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal[sig][mass] = hsignal_apv + hsignal_noapv + hsignal_17 + hsignal_18

#             httbar  = httbar_apv + httbar_noapv + httbar_17 + httbar_18
#             hcontam = hcontam_apv + hcontam_noapv + hcontam_17 + hcontam_18

#             print('filling root files...')
#             if 'nominal' in syst:
#                 syst = ''
#                 catsystString = catname+syst
#                 hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_apv   = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_17 = getHist(variable, 'JetHT', True, '2017',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_17 = getHist(variable, 'JetHT', False, '2017',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_18 = getHist(variable, 'JetHT', True, '2018',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_18 = getHist(variable, 'JetHT', False, '2018',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

#                 hntmj = hntmj_apv + hntmj_noapv + hntmj_17 + hntmj_18
#                 hdata = hdata_apv + hdata_noapv + hdata_17 + hdata_18
#                 hntmj_fixed = hntmj + -1*hcontam
#                 print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                 froot["data_obs_"+catsystString] = hdata
#                 froot["bkgest_"+catsystString] = hntmj_fixed

#             print('TTbar_'+catsystString)
#             froot["TTbar_"+catsystString] = httbar


#             for sig in Signals.keys():

#                 if 'RSGluon' not in sig:
#                     if sig != 'ZPrime1': # ZPrime10, 30 and DM
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else: # ZPrime1
#                         [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
#                 else: # RSGluon
#                     [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                     for mass in Signals[sig]:
#                         froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#             # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#             dytext = ''
#             if 'cen' in cat:
#                 dytext = r'$\Delta y$ < 1.0'
#             elif 'fwd' in cat:
#                 dytext = r'$\Delta y$ > 1.0'

#             btext = ''
#             if '0b' in cat:
#                 btext = '0 b-tags'
#             elif '1b' in cat:
#                 btext = '1 b-tag'
#             elif '2b' in cat:
#                 btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'

#             plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, signal, hsignal['ZPrime1']['3000'], hsignal['ZPrime10']['3000'], hsignal['ZPrime30']['3000'], hsignal['ZPrimeDM']['3000'])
#             plt.show()
            
#         elif '2016all' in IOV:

#             catsystString = catname+'_'+syst

#             httbar_apv        = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_apv       = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_noapv      = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_noapv     = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

#             print('loading signals...')
#             for sig in Signals.keys():
#                 for mass in Signals[sig]:
#                     hsignal_apv   = getHist(variable, sig, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_noapv = getHist(variable, sig, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal[sig][mass] = hsignal_apv + hsignal_noapv

#             httbar  = httbar_apv + httbar_noapv
#             hcontam = hcontam_apv + hcontam_noapv

#             print('filling root files...')
#             if 'nominal' in syst:
#                 syst = ''
#                 catsystString = catname+syst
#                 hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_apv   = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

#                 hntmj = hntmj_apv + hntmj_noapv
#                 hdata = hdata_apv + hdata_noapv
#                 hntmj_fixed = hntmj + -1*hcontam
#                 print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                 froot["data_obs_"+catsystString] = hdata
#                 froot["bkgest_"+catsystString] = hntmj_fixed

#             print('TTbar_'+catsystString)
#             froot["TTbar_"+catsystString] = httbar


#             for sig in Signals.keys():

#                 if 'RSGluon' not in sig:
#                     if sig != 'ZPrime1': # ZPrime10, 30 and DM
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else: # ZPrime1
#                         [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
#                 else: # RSGluon
#                     [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                     for mass in Signals[sig]:
#                         froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#             # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#             dytext = ''
#             if 'cen' in cat:
#                 dytext = r'$\Delta y$ < 1.0'
#             elif 'fwd' in cat:
#                 dytext = r'$\Delta y$ > 1.0'

#             btext = ''
#             if '0b' in cat:
#                 btext = '0 b-tags'
#             elif '1b' in cat:
#                 btext = '1 b-tag'
#             elif '2b' in cat:
#                 btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'

#             plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, signal, hsignal['ZPrime1']['3000'], hsignal['ZPrime10']['3000'], hsignal['ZPrime30']['3000'], hsignal['ZPrimeDM']['3000'])
#             plt.show()
#         else:

#             catsystString = catname+'_'+syst

#             httbar   = getHist(variable, 'TTbar', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam  = getHist(variable, 'TTbar', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

#             print('loading signals...')
#             for sig in Signals.keys():
#                 for mass in Signals[sig]:
#                     hsignal_sigmass   = getHist(variable, sig, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal[sig][mass] = hsignal_sigmass


#             print('filling root files...')
#             if 'nominal' in syst:
#                 syst = ''
#                 catsystString = catname+syst

#                 hntmj = getHist(variable, 'JetHT', True, IOV,   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata = getHist(variable, 'JetHT', False, IOV,  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_fixed = hntmj + -1*hcontam
#                 print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                 froot["data_obs_"+catsystString] = hdata
#                 froot["bkgest_"+catsystString] = hntmj_fixed

#             print('TTbar_'+catsystString)
#             froot["TTbar_"+catsystString] = httbar


#             for sig in Signals.keys():

#                 if 'RSGluon' not in sig:
#                     if sig != 'ZPrime1': # ZPrime10, 30 and DM
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else: # ZPrime1
#                         [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
#                 else: # RSGluon
#                     [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                     for mass in Signals[sig]:
#                         froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#             # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#             dytext = ''
#             if 'cen' in cat:
#                 dytext = r'$\Delta y$ < 1.0'
#             elif 'fwd' in cat:
#                 dytext = r'$\Delta y$ > 1.0'

#             btext = ''
#             if '0b' in cat:
#                 btext = '0 b-tags'
#             elif '1b' in cat:
#                 btext = '1 b-tag'
#             elif '2b' in cat:
#                 btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'

#             plotBackgroundEstimateNoData(variable, hntmj_fixed, httbar, IOV, text, signal, hsignal['ZPrime1']['3000'], hsignal['ZPrime10']['3000'], hsignal['ZPrime30']['3000'], hsignal['ZPrimeDM']['3000'])
#             plt.show()
                    
# froot.close()            
            
            
            